In [1]:

import os
import torch

print("=== GPU ===")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
        print("Memory (GB):", round(
            torch.cuda.get_device_properties(i).total_memory / 1e9, 2
        ))

print("\n=== INPUT FILES ===")
for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for filename in files[:5]:
        print("  ", filename)


=== GPU ===
CUDA available: True
GPU 0: Tesla T4
Memory (GB): 15.64
GPU 1: Tesla T4
Memory (GB): 15.64

=== INPUT FILES ===
/kaggle/input
/kaggle/input/models
/kaggle/input/models/qwen-lm
/kaggle/input/models/qwen-lm/qwen-3
/kaggle/input/models/qwen-lm/qwen-3/transformers
/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base
/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1
   model.safetensors.index.json
   config.json
   merges.txt
   model-00005-of-00005.safetensors
   model-00001-of-00005.safetensors
/kaggle/input/competitions
/kaggle/input/competitions/name-that-llm
   sample_submission.parquet
   train.csv
   test.csv


In [2]:

import pandas as pd

DATA_DIR = "/kaggle/input/competitions/name-that-llm"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample = pd.read_parquet(
    f"{DATA_DIR}/sample_submission.parquet"
)

print("=== DATASET SHAPES ===")
print("Train:", train.shape)
print("Test:", test.shape)

print("\n=== TRAINING COLUMNS ===")
print(train.columns.tolist())

print("\n=== FIRST TRAINING SAMPLE ===")
print(train.iloc[0].to_dict())

print("\n=== MODEL DISTRIBUTION ===")
print(train["model"].value_counts())

print("\n=== SUBMISSION FORMAT ===")
print(sample.head())
print(sample.dtypes)


=== DATASET SHAPES ===
Train: (2699, 4)
Test: (3, 2)

=== TRAINING COLUMNS ===
['id', 'prompt_id', 'text', 'model']

=== FIRST TRAINING SAMPLE ===
{'id': '0018b3765972', 'prompt_id': 'p081', 'text': 'When a language dies, a particular astronomy dims: not only names for stars, but whole grammars of attention that taught speakers how to read the night. Some tongues map the sky by kinship, binding the Milky Way to ancestors, rivers, or the migration paths of birds; others encode lunar phases as verbs, so time is not a line but a recurring breath. A word can carry the exact color of dawn on snow, the etiquette of speaking to thunder, or the mathematics woven into knot calendars and oral star charts. With extinction, those relations collapse into silence. No dictionary can resurrect the sky those speakers inhabited, nor the quiet ethics it taught. We lose metaphors that made the universe intimate, ethical, and alive, cosmologies in which darkness is a listener, comets are warnings carried b

In [3]:

import importlib.metadata as metadata
import torch

print("=== LIBRARY VERSIONS ===")

for package in [
    "torch",
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "trl"
]:
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "NOT INSTALLED")

print("\n=== GPU CAPABILITIES ===")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}")
    print("Compute capability:", props.major, props.minor)

print("\n=== QWEN TOKENIZER ===")

from transformers import AutoTokenizer

MODEL_PATH = (
    "/kaggle/input/models/qwen-lm/qwen-3/"
    "transformers/8b-base/1"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

sample_text = train.iloc[0]["text"]
tokens = tokenizer(sample_text)

print("Tokenizer loaded successfully!")
print("Number of tokens:", len(tokens["input_ids"]))
print("Vocabulary size:", len(tokenizer))
print("Padding token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)


=== LIBRARY VERSIONS ===
torch 2.11.0+cu130
transformers 5.20.2
peft 0.21.2
accelerate 1.15.0
bitsandbytes NOT INSTALLED
trl NOT INSTALLED

=== GPU CAPABILITIES ===
GPU 0: Tesla T4
Compute capability: 7 5
GPU 1: Tesla T4
Compute capability: 7 5

=== QWEN TOKENIZER ===
Tokenizer loaded successfully!
Number of tokens: 229
Vocabulary size: 151669
Padding token: <|endoftext|>
EOS token: <|im_end|>


In [4]:

%pip install -q bitsandbytes


Note: you may need to restart the kernel to use updated packages.


In [5]:
import bitsandbytes as bnb
import torch
from transformers import BitsAndBytesConfig

print("bitsandbytes version:", bnb.__version__)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit configuration created successfully!")
print(quantization_config)

bitsandbytes version: 0.50.2
4-bit configuration created successfully!
BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [6]:

import torch
from transformers import AutoModelForSequenceClassification

MODEL_PATH = (
    "/kaggle/input/models/qwen-lm/qwen-3/"
    "transformers/8b-base/1"
)

# Consistent mapping between model names and class indices
model_names = sorted(train["model"].unique())

label2id = {
    name: i for i, name in enumerate(model_names)
}

id2label = {
    i: name for name, i in label2id.items()
}

print("Number of classes:", len(model_names))

# Load the pretrained model in 4-bit precision
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=18,
    id2label=id2label,
    label2id=label2id,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    local_files_only=True,
)

model.config.pad_token_id = tokenizer.pad_token_id

print("\nModel loaded successfully!")
print("Model type:", type(model).__name__)

# Check GPU memory use
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved = torch.cuda.memory_reserved(i) / 1e9

    print(
        f"GPU {i}: {allocated:.2f} GB allocated, "
        f"{reserved:.2f} GB reserved"
    )


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Number of classes: 18


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model loaded successfully!
Model type: Qwen3ForSequenceClassification
GPU 0: 3.44 GB allocated, 3.54 GB reserved
GPU 1: 1.40 GB allocated, 1.50 GB reserved


In [7]:

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training
)

# Prepare the quantized Qwen model for training
model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

# Define our first experiment's LoRA parameters
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    modules_to_save=["score"],
    bias="none"
)

# Attach trainable adapters
model = get_peft_model(model, lora_config)

print("LoRA adapters attached!\n")

model.print_trainable_parameters()

print("\n=== CLASSIFICATION HEAD ===")
for name, param in model.named_parameters():
    if "score" in name:
        print(name, "Trainable:", param.requires_grad)

print("\n=== GPU MEMORY ===")
import torch

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i)/1e9:.2f} GB allocated"
    )


LoRA adapters attached!

trainable params: 3,907,584 || all params: 7,572,386,816 || trainable%: 0.0516

=== CLASSIFICATION HEAD ===
base_model.model.score.original_module.weight Trainable: False
base_model.model.score.modules_to_save.default.weight Trainable: True

=== GPU MEMORY ===
GPU 0: 4.70 GB allocated
GPU 1: 1.40 GB allocated
